# AIGC 5005 · Module 05 · Part 1
## The estimator API

Companion notebook for the matching part of the Module 05 learning content. Read each explanation, **predict** what the next cell will show, run it, and compare. Change values and run again; you cannot break anything.

**Kernel:** choose the `module05-sklearn` conda environment (top right in VS Code). To make it: `conda create -n module05-sklearn python=3.11 ipykernel`, then `conda activate module05-sklearn` and `pip install -r requirements.txt`. 

## The data
`customers.py` (in this folder) builds 2,000 invented customers. About one in seven has `churned == 1`.

In [1]:
from customers import make_customers

customers = make_customers()
print(customers.shape)
print(customers["churned"].mean().round(3))
print(customers.groupby("contract")["churned"].mean().round(3))
print(customers["monthly_charge"].isna().sum())
customers.head()

(2000, 6)
0.141
contract
month-to-month    0.196
one-year          0.093
two-year          0.046
Name: churned, dtype: float64
86


,tenure_months,monthly_charge,support_calls,contract,payment,churned
0,7,43.02,2,month-to-month,cheque,0
1,56,54.17,2,month-to-month,bank transfer,0
2,48,70.44,0,month-to-month,card,0
3,32,52.43,1,month-to-month,bank transfer,0
4,32,59.43,2,one-year,card,0


## A transformer: StandardScaler
Learned attributes end in an underscore and exist **only after `fit`**.

In [2]:
from sklearn.preprocessing import StandardScaler

X = customers[["tenure_months", "support_calls"]]
y = customers["churned"]

scaler = StandardScaler()
print(hasattr(scaler, "mean_"))      # False, nothing learned yet
scaler.fit(X)
print(hasattr(scaler, "mean_"))      # True
print(scaler.mean_.round(2))
print(scaler.scale_.round(2))

False
True
[36.5   1.52]
[20.88  1.24]


In [3]:
X_scaled = scaler.transform(X)
print(X_scaled.mean(axis=0).round(2), X_scaled.std(axis=0).round(2))

[-0.  0.] [1. 1.]


## A model: LogisticRegression
Hyperparameters are chosen before `fit`. `coef_`, `intercept_` and `classes_` are learned.

In [4]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
print(model.get_params()["C"], model.get_params()["max_iter"])

model.fit(X_scaled, y)
print(model.coef_.round(3), model.intercept_.round(3))
print(model.classes_)
print(model.predict_proba(X_scaled[:3]).round(3))

1.0 100
[[-0.597  0.457]] [-1.998]
[0 1]
[[0.726 0.274]
 [0.915 0.085]
 [0.947 0.053]]


**Read the weights.** A negative weight on tenure: longer-tenured customers are less likely to leave. A positive weight on support calls: customers who call more are more likely to leave.

## clone and set_params
`clone` gives a fresh, unfitted copy with the same settings. Cross-validation and grid search use it internally.

In [5]:
from sklearn.base import clone

fresh = clone(model)
print(hasattr(fresh, "coef_"), hasattr(model, "coef_"))   # False True
fresh.set_params(C=0.1)
print(fresh.C)

False True
0.1


## Try it
1. Print `scaler.mean_` before and after `fit` and confirm it only exists afterwards.
2. Change `C` to `100`, refit a clone, and compare `coef_`. What happens to the weights?